# Graph Theory Project — verification notebook
Python 3 with NetworkX, NumPy and Matplotlib. Every worked example in the report is re-computed here.
Run the cells from top to bottom (Google Colab: `!pip install networkx sympy` if needed).

In [ ]:
import itertools, heapq
import numpy as np
import networkx as nx
import sympy as sp
import matplotlib.pyplot as plt

## 1. Representations and the Handshaking Lemma (Chapter 1, 2, 6)

In [ ]:
G = nx.Graph([('A','B'),('A','C'),('B','C'),('C','D'),('D','E')])   # Fig. 1.3
print('order n =', G.number_of_nodes(), ' size m =', G.number_of_edges())
print('degrees :', dict(G.degree()))
print('sum of degrees =', sum(d for _, d in G.degree()), '= 2m =', 2*G.number_of_edges())
nodes = sorted(G)
A = nx.to_numpy_array(G, nodelist=nodes, dtype=int)
print('adjacency matrix:\n', A)
print('adjacency list  :', {v: sorted(G[v]) for v in nodes})
K3 = nx.complete_graph(3); A3 = nx.to_numpy_array(K3, dtype=int)
print('A^2 =\n', A3@A3, '\ntrace(A^3)/6 =', np.trace(np.linalg.matrix_power(A3,3))//6, '(triangles)')

## 2. Havel–Hakimi test (Section 2.3)

In [ ]:
def havel_hakimi(seq):
    s = sorted(seq, reverse=True)
    while s:
        d = s.pop(0)
        if d > len(s): return False
        for i in range(d):
            s[i] -= 1
            if s[i] < 0: return False
        s.sort(reverse=True)
    return True
for seq in [(4,3,3,2,2), (3,3,3,1)]:
    print(seq, 'graphical' if havel_hakimi(seq) else 'not graphical', nx.is_graphical(list(seq)))

## 3. Euler and Hamilton (Chapter 3, 4)

In [ ]:
K = nx.Graph([('A','B'),('A','D'),('A','C'),('B','C'),('D','C')])   # Euler degree-condition figure
odd = [v for v, d in K.degree() if d % 2]
print('odd vertices:', odd, '| has Euler circuit:', nx.is_eulerian(K), '| has Euler path:', nx.has_eulerian_path(K))
print('Euler path  :', list(nx.eulerian_path(K, source='A')))
# Konigsberg multigraph: A north bank, B south bank, C east region, D island
KB = nx.MultiGraph([('A','D'),('A','D'),('B','D'),('B','D'),('A','C'),('B','C'),('D','C')])
print('Konigsberg degrees:', dict(KB.degree()), '| Euler path exists:', nx.has_eulerian_path(KB))
def has_ham_cycle(g):
    vs = list(g); first = vs[0]
    return any(all(g.has_edge(p[i], p[(i+1) % len(p)]) for i in range(len(p)))
               for p in ((first,)+q for q in itertools.permutations(vs[1:])))
print('K2,3 Hamiltonian?', has_ham_cycle(nx.complete_bipartite_graph(2,3)), '| K4 Hamiltonian?', has_ham_cycle(nx.complete_graph(4)))

## 4. Travelling salesman, brute force (Section 4.4)

In [ ]:
d = {('A','B'):10,('A','C'):15,('A','D'):20,('B','C'):35,('B','D'):25,('C','D'):30}
w = lambda a, b: d.get((a,b), d.get((b,a)))
tours = {}
for p in itertools.permutations('BCD'):
    t = ('A',)+p+('A',)
    tours[t] = sum(w(t[i],t[i+1]) for i in range(4))
for t, c in sorted(tours.items(), key=lambda x: x[1])[:6]: print('-'.join(t), c)
print('optimal =', min(tours.values()), '| distinct tours = (4-1)!/2 =', 3)

## 5. Shortest paths: Dijkstra and Floyd–Warshall (Chapter 6)

In [ ]:
W = nx.Graph()
W.add_weighted_edges_from([('A','B',4),('A','C',2),('B','C',1),('B','D',5),('C','D',8),('C','E',10),('D','E',2)])
def dijkstra(g, src):
    dist = {v: float('inf') for v in g}; dist[src] = 0; pq = [(0, src)]
    while pq:
        dd, u = heapq.heappop(pq)
        if dd > dist[u]: continue
        for v, a in g[u].items():
            if dd + a['weight'] < dist[v]:
                dist[v] = dd + a['weight']; heapq.heappush(pq, (dist[v], v))
    return dist
print('Dijkstra from A :', dijkstra(W,'A'))
print('NetworkX path   :', nx.dijkstra_path(W,'A','E'), nx.dijkstra_path_length(W,'A','E'))
H = nx.DiGraph(); H.add_weighted_edges_from([(1,2,3),(1,4,7),(2,1,8),(2,3,2),(3,1,5),(3,4,1),(4,1,2)])
print('Floyd–Warshall D(4):\n', nx.floyd_warshall_numpy(H, nodelist=[1,2,3,4]).astype(int))

## 6. Spanning trees, Laplacian and Cayley (Chapter 5, 7)

In [ ]:
mst = nx.minimum_spanning_tree(W)
print('MST:', sorted(mst.edges(data='weight')), 'total =', mst.size(weight='weight'))
L = sp.Matrix([[2,-1,-1,0],[-1,2,-1,0],[-1,-1,3,-1],[0,0,-1,1]])      # Fig. 7.3 graph
print('spanning trees by matrix-tree theorem:', L[:3,:3].det())
print('K4 spanning trees:', sp.Matrix(nx.laplacian_matrix(nx.complete_graph(4)).toarray())[1:,1:].det(), '= 4^(4-2) =', 4**2)

## 7. Network flow (Section 4.6.4 and 7.5.1)

In [ ]:
F = nx.DiGraph()
for u,v,c in [('s','a',5),('s','b',4),('a','b',2),('a','t',3),('b','t',5)]: F.add_edge(u,v,capacity=c)
val, flow = nx.maximum_flow(F,'s','t'); cut, parts = nx.minimum_cut(F,'s','t')
print('max flow =', val, '| min cut =', cut, parts)
F2 = nx.DiGraph()
for u,v,c in [('S','A',8),('S','B',5),('A','B',3),('A','T',5),('B','T',8)]: F2.add_edge(u,v,capacity=c)
print('Fig. 4.9 network, max flow =', nx.maximum_flow_value(F2,'S','T'))

## 8. Planarity, colouring and chromatic polynomial (Chapter 5)

In [ ]:
for name, g in [('K5', nx.complete_graph(5)), ('K3,3', nx.complete_bipartite_graph(3,3)), ('K4', nx.complete_graph(4))]:
    print(name, 'planar:', nx.check_planarity(g)[0])
D = nx.Graph([('A','B'),('A','C'),('A','D'),('B','C'),('C','D')])    # diamond used in Section 5.3.3
col = nx.coloring.greedy_color(D, strategy='largest_first'); print('greedy colouring:', col, '| colours used =', len(set(col.values())))
x = sp.symbols('x'); print('chromatic polynomial:', sp.factor(nx.chromatic_polynomial(D)(x) if callable(nx.chromatic_polynomial(D)) else nx.chromatic_polynomial(D)))
print('P(3) =', sp.factor(nx.chromatic_polynomial(D)).subs(x,3))

## 9. BFS, DFS, topological sort, matching (Chapter 6, 5)

In [ ]:
T = nx.Graph([('A','B'),('A','C'),('B','D'),('B','E'),('C','F')])
print('BFS:', ['A']+[v for u,v in nx.bfs_edges(T,'A')]); print('DFS:', ['A']+[v for u,v in nx.dfs_edges(T,'A')])
dag = nx.DiGraph([('A','C'),('B','C'),('C','D'),('C','E'),('D','F'),('E','F')])
print('topological order:', list(nx.topological_sort(dag)))
B = nx.Graph([('S1','P1'),('S1','P2'),('S2','P2'),('S2','P3'),('S3','P1'),('S3','P3')])
print('max matching:', nx.bipartite.maximum_matching(B, top_nodes={'S1','S2','S3'}))

## 9b. Problems added in Chapter 3 and the case studies of Chapter 5

In [ ]:
# Chinese postman, weighted example (Section 3.6.4)
P = nx.Graph(); P.add_weighted_edges_from([('A','B',3),('B','C',4),('C','D',2),('D','A',5),('A','C',6)])
odd = [v for v, d in P.degree() if d % 2]
extra = nx.dijkstra_path_length(P, odd[0], odd[1])
print('odd vertices', odd, '| total length', P.size(weight='weight'), '| repeat shortest path', extra,
      '| postman route =', P.size(weight='weight') + extra)
# Assignment problem (Section 3.10.3)
Cmat = [[9,2,7],[3,6,4],[5,8,1]]
best = min(itertools.permutations(range(3)), key=lambda p: sum(Cmat[i][p[i]] for i in range(3)))
print('assignment', [(f'W{i+1}', f'Task{best[i]+1}') for i in range(3)], 'cost', sum(Cmat[i][best[i]] for i in range(3)))
# Examination timetable by colouring (Case study 1)
T2 = nx.Graph([('M','P'),('M','C'),('P','C'),('M','S'),('C','T'),('E','S'),('E','T')])
col = nx.coloring.greedy_color(T2, strategy='largest_first')
print('timetable slots:', {s: sorted(v for v in col if col[v] == s) for s in sorted(set(col.values()))}, '| slots =', len(set(col.values())))
# Courier route (Case study 2)
dd = {('A','B'):11,('A','C'):22,('A','D'):21,('A','E'):8,('B','C'):15,('B','D'):23,('B','E'):19,('C','D'):24,('C','E'):22,('D','E'):6}
wt = lambda a, b: dd.get((a,b), dd.get((b,a)))
tours = {('A',)+p: sum(wt((('A',)+p+('A',))[i], (('A',)+p+('A',))[i+1]) for i in range(5)) for p in itertools.permutations('BCDE')}
opt = min(tours, key=tours.get); print('courier optimum', '-'.join(opt+('A',)), tours[opt], 'km | distinct tours', len(tours)//2)
cur, seen, tot = 'A', ['A'], 0
while len(seen) < 5:
    nxt = min((v for v in 'ABCDE' if v not in seen), key=lambda v: wt(cur, v)); tot += wt(cur, nxt); cur = nxt; seen.append(cur)
tot += wt(cur, 'A'); print('nearest neighbour', '-'.join(seen+['A']), tot, 'km')

## 10. Drawing a graph with the shortest path highlighted

In [ ]:
pos = nx.spring_layout(W, seed=7)
path = nx.dijkstra_path(W,'A','E'); pe = list(zip(path, path[1:]))
nx.draw(W, pos, with_labels=True, node_color='#dbe9f6', edge_color='#9aa9b5')
nx.draw_networkx_edges(W, pos, edgelist=pe, edge_color='red', width=3)
nx.draw_networkx_edge_labels(W, pos, nx.get_edge_attributes(W,'weight'))
plt.savefig('shortest_path.png', dpi=200); plt.show()